# Лабораторная работа №6
### Предобработка и классификация текста.
Перевощиков Н.Д. ИУ5Ц-21М ММО АСОИУ

# Часть 1. Предобработка текста

In [4]:
# !pip install nltk spacy scikit-learn pandas numpy gensim matplotlib seaborn
# !python -m spacy download en_core_web_sm
# !python -m spacy download ru_core_news_sm

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

import spacy

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from gensim.models import Word2Vec
from gensim.models import FastText

nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Nikistor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nikistor\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Nikistor\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Nikistor\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Nikistor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
text = """Илон Маск основал компанию SpaceX в 2002 году в Калифорнии для разработки космических ракет. 
Основной целью SpaceX является снижение стоимости космических запусков и создание технологий, 
необходимых для колонизации Марса. Одним из ключевых достижений компании стал запуск первой 
частично многоразовой ракеты Falcon 9, которая совершила успешную посадку после вывода полезной нагрузки на орбиту."""

print("ИСХОДНЫЙ ТЕКСТ:")
print(text)

ИСХОДНЫЙ ТЕКСТ:
Илон Маск основал компанию SpaceX в 2002 году в Калифорнии для разработки космических ракет. 
Основной целью SpaceX является снижение стоимости космических запусков и создание технологий, 
необходимых для колонизации Марса. Одним из ключевых достижений компании стал запуск первой 
частично многоразовой ракеты Falcon 9, которая совершила успешную посадку после вывода полезной нагрузки на орбиту.


### 1. Токенизация
Токенизация — это процесс разбиения текста на отдельные слова или символы.

In [7]:
# Токенизация
sentences = sent_tokenize(text, language="russian")
tokens = word_tokenize(text, language="russian")

print("ТОКЕНИЗАЦИЯ")
print("Предложения:", sentences)
print("Токены:", tokens)

# spaCy model
try:
    nlp = spacy.load("ru_core_news_sm")
except:
    nlp = spacy.load("en_core_web_sm")

doc = nlp(text)

ТОКЕНИЗАЦИЯ
Предложения: ['Илон Маск основал компанию SpaceX в 2002 году в Калифорнии для разработки космических ракет.', 'Основной целью SpaceX является снижение стоимости космических запусков и создание технологий, \nнеобходимых для колонизации Марса.', 'Одним из ключевых достижений компании стал запуск первой \nчастично многоразовой ракеты Falcon 9, которая совершила успешную посадку после вывода полезной нагрузки на орбиту.']
Токены: ['Илон', 'Маск', 'основал', 'компанию', 'SpaceX', 'в', '2002', 'году', 'в', 'Калифорнии', 'для', 'разработки', 'космических', 'ракет', '.', 'Основной', 'целью', 'SpaceX', 'является', 'снижение', 'стоимости', 'космических', 'запусков', 'и', 'создание', 'технологий', ',', 'необходимых', 'для', 'колонизации', 'Марса', '.', 'Одним', 'из', 'ключевых', 'достижений', 'компании', 'стал', 'запуск', 'первой', 'частично', 'многоразовой', 'ракеты', 'Falcon', '9', ',', 'которая', 'совершила', 'успешную', 'посадку', 'после', 'вывода', 'полезной', 'нагрузки', 'на', '

### 2. Частеречная разметка
Частеречная разметка определяет часть речи для каждого токена.

In [9]:
# Частеречная разметка
print("ЧАСТЕРЕЧНАЯ РАЗМЕТКА")
for token in doc:
    print(token.text, "->", token.pos_, token.tag_)

ЧАСТЕРЕЧНАЯ РАЗМЕТКА
Илон -> PROPN NNP
Маск -> PROPN NNP
основал -> NOUN NN
компанию -> PROPN NNP
SpaceX -> PROPN NNP
в -> PROPN NNP
2002 -> NUM CD
году -> PROPN NNP
в -> PROPN NNP
Калифорнии -> PROPN NNP
для -> PROPN NNP
разработки -> PROPN NNP
космических -> PROPN NNP
ракет -> NOUN NNS
. -> PUNCT .

 -> SPACE _SP
Основной -> ADJ JJ
целью -> NOUN NN
SpaceX -> PROPN NNP
является -> PROPN NNP
снижение -> PROPN NNP
стоимости -> PROPN NNP
космических -> VERB VBP
запусков -> NOUN NN
и -> DET PDT
создание -> DET DT
технологий -> NOUN NN
, -> PUNCT ,

 -> SPACE _SP
необходимых -> VERB VBZ
для -> PROPN NNP
колонизации -> NOUN NN
Марса -> PROPN NNP
. -> PUNCT .
Одним -> PROPN NNP
из -> PROPN NNP
ключевых -> VERB VBP
достижений -> PROPN NNP
компании -> PROPN NNP
стал -> PROPN NNP
запуск -> PROPN NNP
первой -> NOUN NN

 -> SPACE _SP
частично -> ADJ JJ
многоразовой -> NOUN NN
ракеты -> NOUN NN
Falcon -> PROPN NNP
9 -> NUM CD
, -> PUNCT ,
которая -> NOUN NN
совершила -> PROPN NNP
успешную -> PROPN

### 3. Лемматизация
Лемматизация приводит слова к их базовой форме (лемме).

In [11]:
# Лемматизация
print("ЛЕММАТИЗАЦИЯ")
lemmas = [token.lemma_ for token in doc]
print("Леммы:", lemmas)

ЛЕММАТИЗАЦИЯ
Леммы: ['Илон', 'Маск', 'основал', 'компанию', 'SpaceX', 'в', '2002', 'году', 'в', 'Калифорнии', 'для', 'разработки', 'космических', 'ракет', '.', '\n', 'основной', 'целью', 'SpaceX', 'является', 'снижение', 'стоимости', 'космических', 'запусков', 'и', 'создание', 'технологий', ',', '\n', 'необходимых', 'для', 'колонизации', 'Марса', '.', 'Одним', 'из', 'ключевых', 'достижений', 'компании', 'стал', 'запуск', 'первой', '\n', 'частично', 'многоразовой', 'ракеты', 'Falcon', '9', ',', 'которая', 'совершила', 'успешную', 'посадку', 'после', 'вывода', 'полезной', 'нагрузки', 'на', 'орбиту', '.']


### 4. Выделение именованных сущностей
Именованные сущности — это имена людей, организаций, географических объектов и т.д.

In [13]:
# NER
print("ИМЕНОВАННЫЕ СУЩНОСТИ")
for ent in doc.ents:
    print(ent.text, "->", ent.label_)

ИМЕНОВАННЫЕ СУЩНОСТИ
Илон Маск -> PERSON
2002 -> DATE
Калифорнии -> GPE
Основной -> ORG
SpaceX -> NORP
снижение -> PERSON
создание технологий -> ORG
Марса -> DATE
Одним -> ORG
Falcon 9 -> LAW


### 5. Разбор предложения
Синтаксический разбор показывает структуру предложения.

In [15]:
# Синтаксический разбор
print("СИНТАКСИЧЕСКИЙ РАЗБОР")
for token in doc:
    print(f"{token.text:15} POS = {token.pos_:6} DEP = {token.dep_:12} HEAD = {token.head.text}")

СИНТАКСИЧЕСКИЙ РАЗБОР
Илон            POS = PROPN  DEP = compound     HEAD = Маск
Маск            POS = PROPN  DEP = compound     HEAD = компанию
основал         POS = NOUN   DEP = npadvmod     HEAD = Маск
компанию        POS = PROPN  DEP = compound     HEAD = в
SpaceX          POS = PROPN  DEP = compound     HEAD = в
в               POS = PROPN  DEP = ROOT         HEAD = в
2002            POS = NUM    DEP = nummod       HEAD = в
году            POS = PROPN  DEP = compound     HEAD = в
в               POS = PROPN  DEP = punct        HEAD = в
Калифорнии      POS = PROPN  DEP = compound     HEAD = разработки
для             POS = PROPN  DEP = compound     HEAD = разработки
разработки      POS = PROPN  DEP = compound     HEAD = космических
космических     POS = PROPN  DEP = ROOT         HEAD = космических
ракет           POS = NOUN   DEP = dobj         HEAD = космических
.               POS = PUNCT  DEP = punct        HEAD = космических

               POS = SPACE  DEP = dep          HEAD

# Часть 2. Классификация текстов

In [17]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

### 1. Способ 1: CountVectorizer или TfidfVectorizer
#### Подготовка данных:
Для классификации текстов возьмем набор данных, например, 20 Newsgroups, который доступен в sklearn.

In [19]:
# Загрузка данных
categories = ['sci.space', 'rec.sport.hockey']
data = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))

# Вывод информации о датасете
print(f"Количество текстов: {len(data.data)}")
print(f"Категории: {data.target_names}")
print(f"Метки классов: {set(data.target)}\n")

# Вывод примеров текстов
for i in range(5):  # Выведем первые 5 текстов
    print(f"Текст #{i + 1} (класс: {data.target[i]} - {data.target_names[data.target[i]]}):")
    print(data.data[i][:300])  # Показываем первые 300 символов текста
    print("-" * 80)

Количество текстов: 1986
Категории: ['rec.sport.hockey', 'sci.space']
Метки классов: {0, 1}

Текст #1 (класс: 1 - sci.space):






Well, I guess I'm left wondering just who all the 'light fascists'
think *they* are.  Yes, I understand the issues.  I don't even
particularly care for the idea.  But am I the only one that finds the
sort of overreaction above just a *little* questionable?  You must
find things like the Moon *
--------------------------------------------------------------------------------
Текст #2 (класс: 0 - rec.sport.hockey):




	I think that you are incorrect, Roger.  Patrick,
Smythe and Adams all played or coached in the league before becoming
front office types.  Hence, they did help build the league, although
they were not great players themselves.  

	I agree that a name is a name is a name, and if some people
have
--------------------------------------------------------------------------------
Текст #3 (класс: 1 - sci.space):
Forwarded from Neal Ausman, Galileo M

#### Разделение на обучающую и тестовую выборки.
После загрузки данных, мы можем разделить их на обучающую и тестовую выборки:

In [21]:
# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

# Вывод размеров выборок
print(f"Размер обучающей выборки: {len(X_train)}")
print(f"Размер тестовой выборки: {len(X_test)}")

Размер обучающей выборки: 1588
Размер тестовой выборки: 398


#### Векторизация текста:

In [23]:
# Векторизация текста использование TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Вывод размерности векторизованных данных
print(f"Размерность обучающей выборки после векторизации: {X_train_vec.shape}")
print(f"Размерность тестовой выборки после векторизации: {X_test_vec.shape}")

Размерность обучающей выборки после векторизации: (1588, 5000)
Размерность тестовой выборки после векторизации: (398, 5000)


#### Обучение модели:

In [26]:
# Обучение модели
clf = MultinomialNB()
clf.fit(X_train_vec, y_train)

# Предсказание на тестовой выборке
y_pred_tfidf = clf.predict(X_test_vec)

# Оценка качества
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
prec_tfidf, rec_tfidf, f1_tfidf, _ = precision_recall_fscore_support(
    y_test, y_pred_tfidf, average="weighted", zero_division=0
)

print("TF-IDF (TfidfVectorizer)")
print("Accuracy:", acc_tfidf)
print("Precision:", prec_tfidf)
print("Recall:", rec_tfidf)
print("F1:", f1_tfidf)
print("\nClassification report:")
print(classification_report(y_test, y_pred_tfidf, zero_division=0))

TF-IDF (TfidfVectorizer)
Accuracy: 0.9522613065326633
Precision: 0.9527969459204582
Recall: 0.9522613065326633
F1: 0.9522338522962078

Classification report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95       202
           1       0.97      0.93      0.95       196

    accuracy                           0.95       398
   macro avg       0.95      0.95      0.95       398
weighted avg       0.95      0.95      0.95       398



### 2. Способ 2: Word2Vec, GloVe или FastText
#### Подготовка данных:

In [28]:
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression

# Токенизация текста
sentences = [word_tokenize(doc.lower()) for doc in data.data]

# Обучение Word2Vec
model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4)

# Функция для получения вектора документа
def document_vector(doc):
    vectors = [model.wv[word] for word in doc if word in model.wv]
    return sum(vectors) / len(vectors) if vectors else [0] * 100

# Преобразование текстов в векторы
X_train_w2v = [document_vector(word_tokenize(doc.lower())) for doc in X_train]
X_test_w2v = [document_vector(word_tokenize(doc.lower())) for doc in X_test]

#### Обучение модели:

In [31]:
# Обучение классификатора
clf_w2v = LogisticRegression(max_iter=1000)
clf_w2v.fit(X_train_w2v, y_train)

# Предсказание на тестовой выборке
y_pred_w2v = clf_w2v.predict(X_test_w2v)

# Оценка качества
acc_w2v = accuracy_score(y_test, y_pred_w2v)
prec_w2v, rec_w2v, f1_w2v, _ = precision_recall_fscore_support(
    y_test, y_pred_w2v, average="weighted", zero_division=0
)

print("Word2Vec")
print("Accuracy:", acc_w2v)
print("Precision:", prec_w2v)
print("Recall:", rec_w2v)
print("F1:", f1_w2v)
print("\nClassification report:")
print(classification_report(y_test, y_pred_w2v, zero_division=0))

Word2Vec
Accuracy: 0.8366834170854272
Precision: 0.8369578528803322
Recall: 0.8366834170854272
F1: 0.8366885721829606

Classification report:
              precision    recall  f1-score   support

           0       0.85      0.83      0.84       202
           1       0.83      0.85      0.84       196

    accuracy                           0.84       398
   macro avg       0.84      0.84      0.84       398
weighted avg       0.84      0.84      0.84       398



### 3. Сравнение качества моделей
#### Сравним точность двух подходов:

In [59]:
results = pd.DataFrame({
    "Модель": ["TF-IDF", "Word2Vec"],
    "Accuracy": [acc_tfidf, acc_w2v],
    "Precision": [prec_tfidf, prec_w2v],
    "Recall": [rec_tfidf, rec_w2v],
    "F1-score": [f1_tfidf, f1_w2v],
    "Комментарий": [
        "Лучшая точность и стабильность",
        "Потенциал роста при увеличении данных"
    ]
})

from IPython.display import display, HTML

def highlight_best(s):
    is_max = s == s.max()
    return ['background-color: #d4edda' if v else '' for v in is_max]

display(results.style.apply(highlight_best, subset=["Accuracy", "Precision", "Recall", "F1-score"]))

,Модель,Accuracy,Precision,Recall,F1-score,Комментарий
0,TF-IDF,0.952261,0.952797,0.952261,0.952234,Лучшая точность и стабильность
1,Word2Vec,0.836683,0.836958,0.836683,0.836689,Потенциал роста при увеличении данных
